# Vision lattices: verified digit-image pilot

This notebook reviews three executed CNN/SAE runs.
It verifies finite contexts, saved queries, and model
inference. It does not run CLIP, DINO, or a transcoder.
All source files use the `_codexgen` suffix.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import torch

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Open within the project')
    root = root.parent
sys.path.insert(0, str(root / 'src'))
paper = root / 'texs/sparsesurrs/visionlattices'
evidence = root / 'vision_tokens/digits'
from lattmc.vision.paths_codexgen import load_digit_cache
torch.set_num_threads(2)

## Exact order-theoretic checks

Integer matrices test adjunction, closure, ordinal
scaling, and the spatial counterexample.

In [2]:
from lattmc.vision.audit_codexgen import audit
print(json.dumps(audit(), indent=2))

{
  "contexts": 729,
  "adjunctions": 41472,
  "spatial_queries": 729
}


## Recompute retrieval and verify provenance

Verify hashes, split separation, source labels, and all
150 calibration choices and stored test-score arrays.

In [3]:
from lattmc.vision.verify_codexgen import verify_results
print(json.dumps(verify_results(paper), indent=2))
report = json.loads(
    (evidence / 'results/results_codexgen.json').read_text())
for seed in report['seeds']:
    rows = [r for r in report['retrieval']
            if r['seed'] == seed]
    print('Seed', seed)
    for key in ('graded_ap', 'binary_ap',
                'sae_cosine_ap', 'dense_cosine_ap'):
        print(key, round(np.mean([r[key] for r in rows]), 4))

{
  "verified_queries": 150,
  "verified_artifacts": 10
}
Seed 17
graded_ap 0.3565
binary_ap 0.1183
sae_cosine_ap 0.5396
dense_cosine_ap 0.6644
Seed 29
graded_ap 0.318
binary_ap 0.1247
sae_cosine_ap 0.547
dense_cosine_ap 0.6703
Seed 43
graded_ap 0.3637
binary_ap 0.1329
sae_cosine_ap 0.5148
dense_cosine_ap 0.681


## Reload weights and reproduce feature codes

This checks cached codes against actual inference from
the saved CNN and SAE weights on every image.

In [4]:
from lattmc.vision.models_codexgen import DigitCNN, TopKSAE
digits = load_digit_cache(evidence, 17)
images = torch.tensor(digits['images'][:, None] / 16,
                      dtype=torch.float32)
for seed in report['seeds']:
    weights = torch.load(
        evidence / 'checkpoints'
        / f'digits_seed_{seed}_codexgen.pt',
        map_location='cpu', weights_only=True)
    cnn, sae = DigitCNN(), TopKSAE()
    cnn.load_state_dict(weights['cnn'])
    sae.load_state_dict(weights['sae'])
    cnn.eval()
    sae.eval()
    with torch.no_grad():
        hidden = cnn.features(images)
        sites = hidden.permute(0, 2, 3, 1)
        sites = sites.reshape(-1, 16, 32)
        codes = sae.encode(sites).numpy()
    cache = load_digit_cache(evidence, seed)
    np.testing.assert_array_equal(codes, cache['patches'])
    print('Exact cached-code reproduction:', seed)

Exact cached-code reproduction: 17
Exact cached-code reproduction: 29
Exact cached-code reproduction: 43


## A spatial counterexample

Two separate sites can jointly satisfy an image-level
query without any one site satisfying it.

In [5]:
from lattmc.vision.contexts_codexgen import spatial_extents
patches = np.array([[[2, 0], [0, 2]]])
image, site = spatial_extents(patches, [2, 2])
print('Pooled:', image.tolist())
print('Same site:', site.tolist())

Pooled: [True]
Same site: [False]


## Retraining and foundation-model extensions

From the repository root, use the existing environment:

```sh
PYTHONPATH=src uv run --no-sync python -m \
  lattmc.vision.experiment_codexgen \
  --output /private/tmp/vision-reproduction
```

The manuscript README gives table and PDF commands.
ViT-Prisma, SAEV, and Overcomplete are researched
extension options; none was used for the pilot.
See LIBRARIES.md for verified source links and the
activation-cache contract. Do not identify dictionary
indices across independently trained models.